# Notebook 04 - Custom CNN Architecture Design (PyTorch)

**Objective:** Design and build custom Convolutional Neural Network (CNN) architectures from scratch for plant disease classification.

**Key Components:**
- Custom Sequential CNN model with Conv2d and MaxPool layers
- Model architecture visualization
- Parameter counting and model summary
- Save model architecture for reproducibility

## Import Packages

In [12]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch Libraries
import torch
import torch.nn as nn
from torch.nn import Conv2d, MaxPool2d, Flatten, Linear, Dropout, ReLU, Softmax
from torchsummary import summary
from torchinfo import summary
import IPython

print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.11.0+cu130
GPU Available: False


## Define Paths & Global Constants

In [13]:
# ── Project Paths ───────────────────────────────────────────────────────────────
_nb_path = Path(IPython.get_ipython().run_line_magic('pwd', '') if IPython.get_ipython() else '.').resolve()
PROJECT_ROOT = _nb_path.parent
ARCHITECTURE_DIR = PROJECT_ROOT / 'architecture'

# ── Model Configuration Constants ────────────────────────────────────────────
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3
NUM_CLASSES = 38  # Update based on your dataset

# Architecture hyperparameters
INITIAL_FILTERS = 32
FILTER_MULTIPLIER = 2
KERNEL_SIZE = 3
POOL_SIZE = 2
DROPOUT_RATE = 0.3
DENSE_UNITS = 256

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Input Shape: ({IMG_HEIGHT}, {IMG_WIDTH}, {IMG_CHANNELS})")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Device: {DEVICE}")

Input Shape: (224, 224, 3)
Number of Classes: 38
Device: cpu


## Build Custom CNN Architecture

### Option 1: Baseline CNN with Progressive Depth

In [14]:
class BaselineCNN(nn.Module):
    """
    Baseline CNN architecture for plant disease classification.
    
    Architecture:
    - Block 1: Conv2D(32) → Conv2D(32) → MaxPooling → Dropout
    - Block 2: Conv2D(64) → Conv2D(64) → MaxPooling → Dropout
    - Block 3: Conv2D(128) → Conv2D(128) → MaxPooling → Dropout
    - Flatten + Dense layers with Dropout
    - Output: Softmax activation for multiclass classification
    """
    
    def __init__(self, input_channels=3, num_classes=38, dropout_rate=0.3):
        super(BaselineCNN, self).__init__()
        
        # ── Block 1: 32 filters ─────────────────────────────────────────────
        self.block1 = nn.Sequential(
            Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # ── Block 2: 64 filters ─────────────────────────────────────────────
        self.block2 = nn.Sequential(
            Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # ── Block 3: 128 filters ────────────────────────────────────────────
        self.block3 = nn.Sequential(
            Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # ── Fully Connected Layers ──────────────────────────────────────────
        self.flatten = Flatten()
        
        # Calculate flattened size: (224/8) * (224/8) * 128 = 28*28*128 = 100352
        self.fc1 = Linear(128 * 28 * 28, 256)
        self.relu1 = nn.ReLU(inplace=True)
        self.dropout1 = Dropout(dropout_rate)
        
        self.fc2 = Linear(256, 128)
        self.relu2 = nn.ReLU(inplace=True)
        self.dropout2 = Dropout(dropout_rate)
        
        # ── Output Layer ────────────────────────────────────────────────────
        self.output = Linear(128, num_classes)
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.output(x)
        return x


# Build the baseline model
baseline_model = BaselineCNN(input_channels=IMG_CHANNELS, num_classes=NUM_CLASSES, dropout_rate=DROPOUT_RATE)
baseline_model = baseline_model.to(DEVICE)
print("✅ Baseline CNN model built successfully!")

✅ Baseline CNN model built successfully!


### Option 2: Compact CNN (for faster training)

In [15]:
class CompactCNN(nn.Module):
    """
    Compact CNN for faster training with fewer parameters.
    Suitable for resource-constrained environments.
    """
    
    def __init__(self, input_channels=3, num_classes=38, dropout_rate=0.3):
        super(CompactCNN, self).__init__()
        
        # Block 1: 16 filters
        self.block1 = nn.Sequential(
            Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # Block 2: 32 filters
        self.block2 = nn.Sequential(
            Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # Block 3: 64 filters
        self.block3 = nn.Sequential(
            Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            Dropout(dropout_rate)
        )
        
        # Fully Connected Layers
        self.flatten = Flatten()
        self.fc1 = Linear(64 * 28 * 28, 128)
        self.relu1 = nn.ReLU(inplace=True)
        self.dropout1 = Dropout(dropout_rate)
        
        self.output = Linear(128, num_classes)
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.output(x)
        return x


# Build the compact model
compact_model = CompactCNN(input_channels=IMG_CHANNELS, num_classes=NUM_CLASSES, dropout_rate=DROPOUT_RATE)
compact_model = compact_model.to(DEVICE)
print("✅ Compact CNN model built successfully!")

✅ Compact CNN model built successfully!


## Model Summary & Parameter Analysis

In [16]:
print("\n" + "="*80)
print("BASELINE CNN ARCHITECTURE SUMMARY")
print("="*80)
print(baseline_model)

print("\n" + "="*80)
print("COMPACT CNN ARCHITECTURE SUMMARY")
print("="*80)
print(compact_model)


BASELINE CNN ARCHITECTURE SUMMARY
BaselineCNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout(p=0.3, inplace=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4

## Count and Compare Model Parameters

In [17]:
def count_parameters(model):
    """
    Count total, trainable, and non-trainable parameters in a model.
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params
    
    return {
        'total': total_params,
        'trainable': trainable_params,
        'non_trainable': non_trainable_params
    }


# Calculate parameters for both models
baseline_params = count_parameters(baseline_model)
compact_params = count_parameters(compact_model)

# Display comparison
print("\n📊 MODEL PARAMETER COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'Baseline CNN':<20} {'Compact CNN':<20}")
print("="*60)
print(f"{'Total Parameters':<25} {baseline_params['total']:>19,} {compact_params['total']:>19,}")
print(f"{'Trainable':<25} {baseline_params['trainable']:>19,} {compact_params['trainable']:>19,}")
print(f"{'Non-Trainable':<25} {baseline_params['non_trainable']:>19,} {compact_params['non_trainable']:>19,}")
print("="*60)

# Calculate memory footprint (assuming float32)
baseline_memory_mb = (baseline_params['total'] * 4) / (1024 * 1024)
compact_memory_mb = (compact_params['total'] * 4) / (1024 * 1024)

print(f"\n💾 ESTIMATED MEMORY FOOTPRINT (float32)")
print(f"Baseline CNN: {baseline_memory_mb:.2f} MB")
print(f"Compact CNN: {compact_memory_mb:.2f} MB")
print(f"Reduction: {(1 - compact_memory_mb/baseline_memory_mb)*100:.1f}%")


📊 MODEL PARAMETER COMPARISON
Metric                    Baseline CNN         Compact CNN         
Total Parameters                   26,015,174           6,451,142
Trainable                          26,015,174           6,451,142
Non-Trainable                               0                   0

💾 ESTIMATED MEMORY FOOTPRINT (float32)
Baseline CNN: 99.24 MB
Compact CNN: 24.61 MB
Reduction: 75.2%


## Visualize Model Architecture

In [18]:
# Create sample input for shape inference
# torchinfo can take the shape tuple directly, so we don't always need a tensor
input_size = (1, IMG_CHANNELS, IMG_HEIGHT, IMG_WIDTH)

# Visualize baseline model
try:
    print("\n--- Baseline Model Architecture ---")
    baseline_stats = summary(baseline_model, input_size=input_size, device=DEVICE, verbose=0)
    
    # Save the text summary to a file
    with open(ARCHITECTURE_DIR / 'baseline_cnn_architecture.txt', 'w') as f:
        f.write(str(baseline_stats))
    
    print(baseline_stats)
    print(f"✅ Baseline model summary saved to: {ARCHITECTURE_DIR / 'baseline_cnn_architecture.txt'}")
except Exception as e:
    print(f"⚠️ Could not summarize baseline model: {e}")

# Visualize compact model
try:
    print("\n--- Compact Model Architecture ---")
    compact_stats = summary(compact_model, input_size=input_size, device=DEVICE, verbose=0)
    
    # Save the text summary to a file
    with open(ARCHITECTURE_DIR / 'compact_cnn_architecture.txt', 'w') as f:
        f.write(str(compact_stats))
        
    print(compact_stats)
    print(f"✅ Compact model summary saved to: {ARCHITECTURE_DIR / 'compact_cnn_architecture.txt'}")
except Exception as e:
    print(f"⚠️ Could not summarize compact model: {e}")



--- Baseline Model Architecture ---
Layer (type:depth-idx)                   Output Shape              Param #
BaselineCNN                              [1, 38]                   --
├─Sequential: 1-1                        [1, 32, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 32, 224, 224]         896
│    └─ReLU: 2-2                         [1, 32, 224, 224]         --
│    └─Conv2d: 2-3                       [1, 32, 224, 224]         9,248
│    └─ReLU: 2-4                         [1, 32, 224, 224]         --
│    └─MaxPool2d: 2-5                    [1, 32, 112, 112]         --
│    └─Dropout: 2-6                      [1, 32, 112, 112]         --
├─Sequential: 1-2                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-7                       [1, 64, 112, 112]         18,496
│    └─ReLU: 2-8                         [1, 64, 112, 112]         --
│    └─Conv2d: 2-9                       [1, 64, 112, 112]         36,928
│    └─ReLU: 2-10                   

## Save Model Architectures

In [19]:
# Save model architecture as string representation
baseline_config_path = ARCHITECTURE_DIR / 'baseline_model_config.txt'
with open(baseline_config_path, 'w') as f:
    f.write(str(baseline_model))
print(f"✅ Baseline model config saved to: {baseline_config_path}")

compact_config_path = ARCHITECTURE_DIR / 'compact_model_config.txt'
with open(compact_config_path, 'w') as f:
    f.write(str(compact_model))
print(f"✅ Compact model config saved to: {compact_config_path}")

# Save model state dict (for loading later)
baseline_state_path = ARCHITECTURE_DIR / 'baseline_model_state.pth'
torch.save(baseline_model.state_dict(), baseline_state_path)
print(f"✅ Baseline model state saved to: {baseline_state_path}")

compact_state_path = ARCHITECTURE_DIR / 'compact_model_state.pth'
torch.save(compact_model.state_dict(), compact_state_path)
print(f"✅ Compact model state saved to: {compact_state_path}")

# Save model metadata
metadata = {
    'framework': 'PyTorch',
    'input_shape': (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
    'num_classes': NUM_CLASSES,
    'baseline_params': baseline_params,
    'compact_params': compact_params,
    'baseline_memory_mb': baseline_memory_mb,
    'compact_memory_mb': compact_memory_mb
}

metadata_path = ARCHITECTURE_DIR / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Model metadata saved to: {metadata_path}")

✅ Baseline model config saved to: /home/tech-angel/PycharmProjects/agrolens-ai/architecture/baseline_model_config.txt
✅ Compact model config saved to: /home/tech-angel/PycharmProjects/agrolens-ai/architecture/compact_model_config.txt
✅ Baseline model state saved to: /home/tech-angel/PycharmProjects/agrolens-ai/architecture/baseline_model_state.pth
✅ Compact model state saved to: /home/tech-angel/PycharmProjects/agrolens-ai/architecture/compact_model_state.pth
✅ Model metadata saved to: /home/tech-angel/PycharmProjects/agrolens-ai/architecture/model_metadata.json
